# HalalifyAI: repeatable stock and SEC data cleaning

This notebook contains the complete cleaner. Use **Run All** after setting the project folder below. It reads local downloads only and never accesses `.env` or requests API credentials.

It checks input hashes, reconciles raw price responses, validates OHLC and exchange sessions, preserves SEC reporting units and filing versions, and repeats safe transformations until each table stops changing. Unresolved evidence stays in review reports. Convergence means the automatic rules have stabilized, not that the data is ready for backtesting.

**Your inspected downloads:** 127,501 prices for 50 companies plus SPY, and 100 SEC response files. LIN starts on 2018-10-31. The other price series start on 2016-01-04. Missing dividend, split-event and bar-availability fields are expected collector omissions and are not replaced with zero.


## 1. Install dependencies
Run once in the notebook kernel. Internet is needed for installation only. The cleaning run is offline.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'pandas>=2.2,<4', 'numpy>=1.26,<3',
                       'exchange-calendars>=4.5,<5', 'tzdata>=2024.1'])


## 2. Choose your folder and snapshots
Local Jupyter: start the notebook in the downloader folder. Otherwise replace `ROOT` with the full folder path.

Colab: copy the downloader's `data/raw/` and `data/interim/` directories to Google Drive, mount Drive below, and point `ROOT` to that folder. The original relative directory structure must be intact. The notebook itself contains the cleaner, so you do not need to upload the Python script. Cleaned output is written beneath `ROOT/data/cleaned/`.

The two snapshot versions below pin the downloads inspected for this notebook. For a later download, change those names or set them to `None` to choose the latest snapshot. A failed latest snapshot raises an error rather than silently using older data.


In [ ]:
from pathlib import Path
# Colab only: uncomment these lines.
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = Path('/content/drive/MyDrive/HalalifyAI-Local-Downloader')
ROOT = Path.cwd()
PRICE_VERSION = 'alpaca_sip_20261009T110617242394Z'
SEC_VERSION = '20261009T110657727507Z'
START = '2016-01-01'
END = '2025-12-31'
MAX_PASSES = 6
LARGE_RETURN_THRESHOLD = 0.25  # Review flag only. No clipping or deletion.
assert (ROOT / 'data/raw/alpaca').is_dir(), f'Please set ROOT to your downloader folder: {ROOT}'
assert (ROOT / 'data/raw/sec_snapshots').is_dir(), 'SEC snapshot folder missing'
print('Project:', ROOT.resolve())


## 3. Load the complete cleaning engine
This cell defines the functions. It does not modify the data. Company facts are handled one issuer at a time to limit memory usage.


In [ ]:
"""Repeatable, offline cleaning of this HalalifyAI downloader's snapshots.

Run: python clean_data.py --root .
Raw inputs are immutable. Every execution writes a new versioned output directory.
"""
from __future__ import annotations
import argparse
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import sys
import numpy as np
import pandas as pd

PRICE_NUMERIC = ['open', 'high', 'low', 'close', 'volume', 'adjusted_close']
PRICE_KEYS = ['ticker', 'session_date']
FACT_COLUMNS = ['ticker', 'cik', 'entity_name', 'taxonomy', 'concept', 'label',
                'unit', 'start', 'end', 'val', 'accn', 'fy', 'fp', 'form', 'filed',
                'frame', 'source_file', 'source_row']
FACT_KEY = ['cik', 'taxonomy', 'concept', 'unit', 'start', 'end', 'accn', 'filed']

@dataclass
class Config:
    root: str = '.'
    price_version: str | None = None
    sec_version: str | None = None
    start: str | None = None
    end: str | None = None
    max_passes: int = 6
    return_review_threshold: float = 0.25
    allow_spy_calendar_proxy: bool = False


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def fingerprint(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False, default=str), encoding='utf-8')


def choose(base, version):
    if version:
        p = base / version
        if not (p / 'metadata.json').is_file():
            raise FileNotFoundError(p / 'metadata.json')
        return p
    paths = sorted(p.parent for p in base.glob('*/metadata.json'))
    if not paths:
        raise FileNotFoundError(f'No snapshot under {base}')
    # Choose latest explicitly, then validate it. Never silently fall back after failure.
    return paths[-1]


def checked_date(series):
    # Downloader and SEC schemas use ISO calendar dates, not locale-dependent dates.
    s = series.astype('string')
    parsed = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    valid = s.str.fullmatch(r'\d{4}-\d{2}-\d{2}').fillna(False) & parsed.notna()
    return parsed.where(valid)


def clean_loop(frame, transform, label, history, max_passes):
    if max_passes < 2:
        raise ValueError('max_passes must be at least 2 to prove stability.')
    work = frame.copy()
    rejected = []
    for pass_number in range(1, max_passes + 1):
        before = fingerprint(work)
        n = len(work)
        work, bad = transform(work)
        work = work.reset_index(drop=True)
        if not bad.empty:
            bad = bad.copy()
            bad['cleaning_pass'] = pass_number
            rejected.append(bad)
        after = fingerprint(work)
        history.append(dict(table=label, pass_number=pass_number, rows_before=n,
                            rows_after=len(work), removed=len(bad), stable=before == after))
        if before == after:
            return work, pd.concat(rejected, ignore_index=True) if rejected else pd.DataFrame()
    raise RuntimeError(f'{label} did not converge after {max_passes} passes. Review rules.')


def remove_bad(df, reasons):
    bad_mask = reasons.ne('')
    bad = df.loc[bad_mask].copy()
    bad['rejection_reason'] = reasons.loc[bad_mask]
    return df.loc[~bad_mask].copy(), bad


def add_reason(reasons, mask, reason):
    mask = pd.Series(mask, index=reasons.index).fillna(False)
    reasons.loc[mask] = reasons.loc[mask].map(lambda x: x + ';' if x else '') + reason


def price_pass(df, start, end):
    df = df.copy()
    df['ticker'] = df.ticker.astype('string').str.strip().str.upper()
    for col in PRICE_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
    dates = checked_date(df.session_date)
    df['session_date'] = dates.dt.strftime('%Y-%m-%d').astype('string')
    reasons = pd.Series('', index=df.index, dtype='string')
    add_reason(reasons, df.ticker.isna() | df.ticker.eq(''), 'missing_ticker')
    add_reason(reasons, dates.isna(), 'invalid_session_date')
    add_reason(reasons, dates.lt(pd.Timestamp(start)) | dates.gt(pd.Timestamp(end)), 'outside_requested_window')
    add_reason(reasons, ~np.isfinite(df[PRICE_NUMERIC]).all(axis=1), 'missing_or_nonfinite_numeric')
    add_reason(reasons, df[['open','high','low','close','adjusted_close']].le(0).any(axis=1), 'nonpositive_price')
    add_reason(reasons, df.volume.lt(0) | df.volume.mod(1).ne(0), 'invalid_volume')
    add_reason(reasons, df.high.lt(df[['open','low','close']].max(axis=1)) |
               df.low.gt(df[['open','high','close']].min(axis=1)), 'invalid_ohlc')
    # Remove byte-equivalent observations first; conflicting keys are all quarantined.
    compare = [c for c in df.columns if c != 'source_row']
    exact = df.duplicated(compare, keep='first')
    add_reason(reasons, exact, 'exact_duplicate')
    survivors = df.loc[~exact]
    conflicts = survivors.duplicated(PRICE_KEYS, keep=False)
    add_reason(reasons, df.index.isin(survivors.index[conflicts]), 'conflicting_ticker_date')
    good, bad = remove_bad(df, reasons)
    return good.sort_values(PRICE_KEYS, kind='stable'), bad


def fact_pass(df):
    df = df.copy()
    df['val'] = pd.to_numeric(df.val, errors='coerce').astype('float64')
    reasons = pd.Series('', index=df.index, dtype='string')
    for col in ['end', 'filed']:
        parsed = checked_date(df[col])
        add_reason(reasons, parsed.isna(), f'invalid_{col}')
        df[col] = parsed.dt.strftime('%Y-%m-%d').astype('string')
    starts = checked_date(df.start)
    add_reason(reasons, df.start.notna() & starts.isna(), 'invalid_start')
    df['start'] = starts.dt.strftime('%Y-%m-%d').astype('string')
    add_reason(reasons, starts.gt(checked_date(df.end)), 'reversed_period')
    add_reason(reasons, ~np.isfinite(df.val), 'missing_or_nonfinite_value')
    for col in ['accn','concept','unit','cik']:
        add_reason(reasons, df[col].isna() | df[col].astype('string').str.strip().eq(''), f'missing_{col}')
    compare = [c for c in FACT_COLUMNS if c not in ['source_file','source_row']]
    exact = df.duplicated(compare, keep='first')
    add_reason(reasons, exact, 'exact_duplicate')
    # Preserve repeated periods in different filings and distinct units/taxonomies.
    survivors = df.loc[~exact]
    conflicts = survivors.groupby(FACT_KEY, dropna=False).val.transform('nunique').gt(1)
    add_reason(reasons, df.index.isin(survivors.index[conflicts]), 'conflicting_fact_same_filing')
    good, bad = remove_bad(df, reasons)
    return good.sort_values(FACT_KEY + ['source_row'], kind='stable', na_position='first'), bad


def append_csv(df, path):
    if df.empty:
        return
    df.to_csv(path, mode='a', header=not path.exists(), index=False)


def verify_inputs(price_dir, sec_dir, pmeta, smeta, price_csv):
    checks = []
    def check(p, expected):
        if not p.is_file():
            raise FileNotFoundError(p)
        actual = digest(p)
        checks.append(dict(file=str(p), expected_sha256=expected, actual_sha256=actual))
        if not expected or actual != expected:
            raise ValueError(f'Missing/mismatched checksum: {p}')
    check(price_csv, pmeta.get('normalized_sha256'))
    for item in pmeta.get('pages', []):
        check(price_dir / item['file'], item.get('sha256'))
    if not pmeta.get('pages'):
        raise ValueError('Price metadata has no raw pages.')
    check(sec_dir / 'company_tickers.json', smeta.get('ticker_mapping_sha256'))
    for item in smeta.get('files', []):
        check(sec_dir / item['file'], item.get('sha256'))
    for p in [price_dir/'metadata.json', sec_dir/'metadata.json']:
        checks.append(dict(file=str(p), expected_sha256=None, actual_sha256=digest(p)))
    return checks


def reconcile_raw(prices, price_dir, pmeta):
    from zoneinfo import ZoneInfo
    for basis in ['split', 'all']:
        rows = []
        for item in pmeta['pages']:
            if not item['file'].startswith(basis + '_'):
                continue
            payload = read_json(price_dir / item['file'])
            for ticker, bars in (payload.get('bars') or {}).items():
                for bar in bars:
                    ts = pd.Timestamp(bar['t'])
                    if ts.tzinfo is None:
                        raise ValueError('Raw timestamp lacks timezone.')
                    rows.append(dict(ticker=ticker, session_date=ts.tz_convert(ZoneInfo('America/New_York')).strftime('%Y-%m-%d'),
                                     open=bar['o'], high=bar['h'], low=bar['l'], close=bar['c'], volume=bar['v']))
        raw = pd.DataFrame(rows)
        if raw.empty or raw.duplicated(PRICE_KEYS).any():
            raise ValueError(f'{basis} raw pages empty or duplicated; inspect raw evidence.')
        merged = prices.merge(raw, on=PRICE_KEYS, how='outer', suffixes=('', '_raw'), indicator=True, validate='one_to_one')
        if not merged['_merge'].eq('both').all():
            raise ValueError(f'{basis} raw and interim session keys differ.')
        pairs = [('adjusted_close', 'close_raw')] if basis == 'all' else [(c, c+'_raw') for c in ['open','high','low','close','volume']]
        for left, right in pairs:
            if not np.isclose(merged[left], merged[right], rtol=1e-12, atol=1e-10, equal_nan=False).all():
                raise ValueError(f'Interim column {left} disagrees with raw {basis} bars.')


def calendar_sessions(start, end, prices, allow_proxy):
    try:
        import exchange_calendars as xc
    except ImportError:
        if not allow_proxy:
            raise RuntimeError('Install requirements-cleaning.txt. Calendar verification is required by default.')
        spy = prices.loc[prices.ticker.eq('SPY'), 'session_date']
        if spy.empty:
            raise ValueError('No SPY data for explicitly requested calendar proxy.')
        return pd.DatetimeIndex(pd.to_datetime(spy.unique())).sort_values(), 'SPY_proxy_unverified'
    cal = xc.get_calendar('XNYS', start=pd.Timestamp(start)-pd.Timedelta(days=10), end=pd.Timestamp(end)+pd.Timedelta(days=10))
    dates = cal.sessions_in_range(start, end)
    return dates.tz_localize(None).normalize(), 'XNYS_exchange_calendars'


def coverage_report(prices, sessions, symbols):
    records, gaps = [], []
    for ticker in symbols:
        d = pd.DatetimeIndex(pd.to_datetime(prices.loc[prices.ticker.eq(ticker),'session_date']))
        missing = sessions.difference(d)
        extra = d.difference(sessions)
        records.append(dict(ticker=ticker, rows=len(d), first_date=str(d.min().date()) if len(d) else None,
                            last_date=str(d.max().date()) if len(d) else None,
                            missing_requested_sessions=len(missing), unexpected_sessions=len(extra)))
        for date in missing:
            region = 'entire_series_missing' if not len(d) else ('before_first_observation' if date < d.min() else ('after_last_observation' if date > d.max() else 'internal_gap'))
            gaps.append(dict(ticker=ticker, session_date=date.strftime('%Y-%m-%d'), reason=region))
    return pd.DataFrame(records), pd.DataFrame(gaps, columns=['ticker','session_date','reason'])


def run(config: Config):
    root = Path(config.root).expanduser().resolve()
    price_dir = choose(root/'data/raw/alpaca', config.price_version)
    sec_dir = choose(root/'data/raw/sec_snapshots', config.sec_version)
    pmeta, smeta = read_json(price_dir/'metadata.json'), read_json(sec_dir/'metadata.json')
    if pmeta.get('status') != 'collected_requires_action_and_calendar_review':
        raise ValueError(f'Price download incomplete: {pmeta.get("status")}')
    if smeta.get('failed') or smeta.get('status') != 'downloaded_requires_identity_review':
        raise ValueError('SEC snapshot incomplete. Inspect its metadata or choose an explicit complete version.')
    price_csv = root/'data/interim/prices'/price_dir.name/'daily_prices.csv'
    start, end = config.start or pmeta['requested_start'], config.end or pmeta['requested_end']
    if pd.Timestamp(start) > pd.Timestamp(end):
        raise ValueError('start must precede end')
    if start < pmeta['requested_start'] or end > pmeta['requested_end']:
        raise ValueError('Cleaning window must be inside the downloaded price window.')
    version = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = root/'data/cleaned'/version
    out.mkdir(parents=True, exist_ok=False)
    history = []
    try:
        inputs = verify_inputs(price_dir, sec_dir, pmeta, smeta, price_csv)
        write_json(out/'input_manifest.json', inputs)
        raw_prices = pd.read_csv(price_csv, dtype={'ticker':'string','session_date':'string'})
        required = set(PRICE_KEYS + PRICE_NUMERIC + ['adjustment_verified','data_quality_flags'])
        if not required.issubset(raw_prices.columns):
            raise ValueError(f'Missing price columns: {required-set(raw_prices.columns)}')
        reconcile_raw(raw_prices, price_dir, pmeta)
        symbols = sorted(pmeta['requested_symbols'])
        if set(raw_prices.ticker) != set(symbols):
            raise ValueError('Requested and returned symbol sets differ.')
        raw_prices['source_row'] = np.arange(2, len(raw_prices)+2)
        prices, rejected = clean_loop(raw_prices, lambda d: price_pass(d, start, end), 'prices', history, config.max_passes)
        rejected.reindex(columns=list(raw_prices.columns)+['rejection_reason','cleaning_pass']).to_csv(out/'prices_quarantine.csv', index=False)
        if prices.empty:
            raise ValueError('No prices survived validation.')
        sessions, calendar = calendar_sessions(start, end, prices, config.allow_spy_calendar_proxy)
        unexpected = ~pd.to_datetime(prices.session_date).isin(sessions)
        if unexpected.any():
            bad = prices.loc[unexpected].copy()
            bad['rejection_reason'] = 'not_exchange_session'
            bad['cleaning_pass'] = 'calendar'
            append_csv(bad.reindex(columns=list(raw_prices.columns)+['rejection_reason','cleaning_pass']), out/'prices_quarantine.csv')
            prices = prices.loc[~unexpected].copy()
        coverage, gaps = coverage_report(prices, sessions, symbols)
        coverage.to_csv(out/'price_coverage.csv', index=False)
        gaps.to_csv(out/'missing_sessions.csv', index=False)
        # Returns only across consecutive exchange sessions; no bridging removed/missing bars.
        session_number = pd.Series(np.arange(len(sessions)), index=sessions.strftime('%Y-%m-%d'))
        prices['session_number'] = prices.session_date.map(session_number)
        consecutive = prices.groupby('ticker').session_number.diff().eq(1)
        prices['split_close_return'] = prices.groupby('ticker').close.pct_change(fill_method=None).where(consecutive)
        prices['provider_all_adjusted_return'] = prices.groupby('ticker').adjusted_close.pct_change(fill_method=None).where(consecutive)
        prices['adjustment_factor'] = prices.adjusted_close / prices.close
        prices['large_return_review'] = prices.provider_all_adjusted_return.abs().gt(config.return_review_threshold)
        prices['zero_volume_review'] = prices.volume.eq(0)
        prices['identity_history_verified'] = False
        prices['adjustment_verified'] = False
        prices['core_ready'] = False
        prices['data_quality_flags'] = prices.data_quality_flags.fillna('') + ';identity_history_unverified;bar_availability_unverified'
        prices.loc[prices.large_return_review, 'data_quality_flags'] += ';large_return_review'
        prices.loc[prices.zero_volume_review, 'data_quality_flags'] += ';zero_volume_review'
        prices.drop(columns='session_number', inplace=True)
        prices.to_csv(out/'prices_clean.csv', index=False)
        prices[prices.ticker.eq('SPY')].to_csv(out/'spy_clean.csv', index=False)
        prices[prices.large_return_review | prices.zero_volume_review].to_csv(out/'price_review.csv', index=False)
        del raw_prices
        current_mapping = {x['ticker']: str(x['cik_str']).zfill(10) for x in read_json(sec_dir/'company_tickers.json').values()}
        groups = {}
        for item in smeta['files']:
            groups.setdefault(item['ticker'], {})[item['kind']] = item
        expected_companies = set(symbols)-{'SPY'}
        identities, counts, older_files = [], [], []
        fact_total = fact_study = fact_bad = fact_future = 0
        for ticker, files in sorted(groups.items()):
            if not {'companyfacts','submissions'}.issubset(files):
                raise ValueError(f'{ticker}: missing companyfacts/submissions pair')
            print(f'Cleaning SEC {ticker}...', flush=True)
            facts_item, sub_item = files['companyfacts'], files['submissions']
            facts = read_json(sec_dir/facts_item['file'])
            sub = read_json(sec_dir/sub_item['file'])
            cik = str(facts['cik']).zfill(10)
            if cik != str(sub['cik']).zfill(10) or cik != facts_item['cik'] or cik != sub_item['cik']:
                raise ValueError(f'{ticker}: SEC issuer CIK mismatch')
            identities.append(dict(ticker=ticker,cik=cik,entity_name=facts.get('entityName'),
                current_mapping_cik=current_mapping.get(ticker),current_mapping_matches=current_mapping.get(ticker)==cik,
                ticker_in_current_submissions=ticker in sub.get('tickers',[]),
                former_names=json.dumps(sub.get('formerNames',[])),identity_history_verified=False,
                review_reason='historical_mapping_requires_evidence' + (';fixed_XOM_predecessor_mapping' if ticker=='XOM' else '')))
            recent = sub.get('filings',{}).get('recent',{})
            lengths = {len(v) for v in recent.values()}
            if len(lengths)>1:
                raise ValueError(f'{ticker}: ragged submissions arrays')
            filings = pd.DataFrame(recent)
            if 'accessionNumber' not in filings or 'acceptanceDateTime' not in filings:
                raise ValueError(f'{ticker}: missing submission keys')
            if filings.accessionNumber.duplicated().any():
                raise ValueError(f'{ticker}: duplicate submission accession')
            filings.insert(0,'ticker',ticker)
            filings.insert(1,'cik',cik)
            submission_columns = ['ticker','cik','accessionNumber','filingDate','reportDate',
                                  'acceptanceDateTime','act','form','fileNumber','filmNumber',
                                  'items','core_type','size','isXBRL','isInlineXBRL',
                                  'isXBRLNumeric','primaryDocument','primaryDocDescription']
            extra_columns = sorted(set(filings.columns)-set(submission_columns))
            filings['extra_fields_json'] = filings[extra_columns].apply(lambda row: json.dumps(row.to_dict()),axis=1) if extra_columns else '{}'
            append_csv(filings.reindex(columns=submission_columns+['extra_fields_json']), out/'sec_submissions.csv')
            for old in sub.get('filings',{}).get('files',[]):
                older_files.append(dict(ticker=ticker,cik=cik,**old))
            acceptance = pd.to_datetime(filings.acceptanceDateTime, errors='coerce', utc=True, format='mixed')
            acc_map = dict(zip(filings.accessionNumber, acceptance))
            rows = []
            row_id = 0
            for taxonomy, concepts in facts.get('facts',{}).items():
                for concept, content in concepts.items():
                    for unit, observations in content.get('units',{}).items():
                        for obs in observations:
                            row_id += 1
                            rows.append(dict(ticker=ticker,cik=cik,entity_name=facts.get('entityName'),taxonomy=taxonomy,
                                concept=concept,label=content.get('label'),unit=unit,
                                **{k:obs.get(k) for k in ['start','end','val','accn','fy','fp','form','filed','frame']},
                                source_file=facts_item['file'],source_row=row_id))
            frame = pd.DataFrame(rows, columns=FACT_COLUMNS)
            cleaned, bad = clean_loop(frame, fact_pass, 'sec_facts_'+ticker, history, config.max_passes)
            append_csv(bad.reindex(columns=FACT_COLUMNS+['rejection_reason','cleaning_pass']), out/'sec_facts_quarantine.csv')
            cleaned['period_days'] = (checked_date(cleaned.end)-checked_date(cleaned.start)).dt.days + 1
            cleaned['acceptance_at_utc'] = pd.to_datetime(cleaned.accn.map(acc_map), errors='coerce', utc=True)
            # A date-only filed value does not justify assuming availability at midnight.
            fallback = (checked_date(cleaned.filed)+pd.Timedelta(days=1)).dt.tz_localize('America/New_York').dt.tz_convert('UTC')
            cleaned['available_at_utc'] = cleaned.acceptance_at_utc.fillna(fallback)
            cleaned['availability_basis'] = np.where(cleaned.acceptance_at_utc.notna(), 'SEC_acceptance_timestamp', 'conservative_next_calendar_day_EST_EDT')
            cleaned['identity_history_verified'] = False
            cleaned['future_period_review'] = checked_date(cleaned.end).gt(checked_date(cleaned.filed))
            cleaned['data_quality_flags'] = 'historical_identity_unverified;current_snapshot_historical_values_may_be_revised'
            cleaned.loc[cleaned.future_period_review, 'data_quality_flags'] += ';period_after_filing_may_be_prospective_disclosure'
            append_csv(cleaned, out/'sec_facts_clean_long.csv')
            # Study cutoffs exclude future filings even if they report older periods.
            cutoff = (pd.Timestamp(end)+pd.Timedelta(days=1)).tz_localize('America/New_York').tz_convert('UTC')
            study = cleaned[cleaned.end.between(start,end) & cleaned.available_at_utc.lt(cutoff) & ~cleaned.future_period_review]
            append_csv(study, out/'sec_facts_study_asof.csv')
            fact_total += len(cleaned)
            fact_study += len(study)
            fact_bad += len(bad)
            fact_future += int(cleaned.future_period_review.sum())
            counts.append(dict(ticker=ticker,raw_facts=len(frame),clean_facts=len(cleaned),study_asof_facts=len(study),quarantined=len(bad),future_period_reviews=int(cleaned.future_period_review.sum()),
                               missing_acceptance_timestamp=int(cleaned.acceptance_at_utc.isna().sum())))
        for filename, cols in [('sec_facts_quarantine.csv',FACT_COLUMNS+['rejection_reason','cleaning_pass']),
                               ('sec_facts_clean_long.csv',FACT_COLUMNS+['period_days','acceptance_at_utc','available_at_utc','availability_basis','identity_history_verified','future_period_review','data_quality_flags']),
                               ('sec_facts_study_asof.csv',FACT_COLUMNS+['period_days','acceptance_at_utc','available_at_utc','availability_basis','identity_history_verified','future_period_review','data_quality_flags'])]:
            if not (out/filename).exists():
                pd.DataFrame(columns=cols).to_csv(out/filename,index=False)
        pd.DataFrame(identities).to_csv(out/'issuer_identity_review.csv',index=False)
        pd.DataFrame(counts).to_csv(out/'sec_coverage.csv',index=False)
        pd.DataFrame(older_files,columns=['ticker','cik','name','filingCount','filingFrom','filingTo']).to_csv(out/'sec_older_submission_files_not_downloaded.csv',index=False)
        pd.DataFrame(history).to_csv(out/'cleaning_passes.csv',index=False)
        missing_sec = sorted(expected_companies-set(groups))
        extra_sec = sorted(set(groups)-expected_companies)
        report = dict(status='completed_with_review_required',core_ready=False,output_dir=str(out),
            config=asdict(config),price_snapshot=price_dir.name,sec_snapshot=sec_dir.name,start=start,end=end,
            calendar=calendar,input_checksums_verified=True,raw_price_reconciliation=True,
            clean_price_rows=len(prices),price_rows_quarantined=int(sum(x['removed'] for x in history if x['table']=='prices')+unexpected.sum()),
            price_symbols=int(prices.ticker.nunique()),missing_requested_sessions=int(len(gaps)),
            large_return_reviews=int(prices.large_return_review.sum()),zero_volume_reviews=int(prices.zero_volume_review.sum()),
            sec_companies=len(groups),missing_sec_symbols=missing_sec,extra_sec_symbols=extra_sec,
            clean_sec_facts=fact_total,study_asof_sec_facts=fact_study,quarantined_sec_facts=fact_bad,sec_future_period_reviews=fact_future,
            older_submission_files_not_downloaded=len(older_files),all_tables_converged=True,
            unresolved=['Corporate actions need independent evidence.', 'Historical ticker/issuer links need dated identity evidence.',
                        'Bar availability timestamps were not collected.', 'Missing sessions need listing-history or provider evidence.',
                        'Older SEC submission chunks were referenced but not downloaded.',
                        'SEC snapshots are current extracts, not archived point-in-time vintages.'],
            environment=dict(python=platform.python_version(),pandas=pd.__version__,numpy=np.__version__,
                             exchange_calendars=importlib.metadata.version('exchange-calendars') if calendar.startswith('XNYS') else None))
        # Verify invariants independently after the loop.
        assert not prices.duplicated(PRICE_KEYS).any()
        assert np.isfinite(prices[PRICE_NUMERIC]).all().all()
        assert prices.session_date.between(start,end).all()
        write_json(out/'report.json', report)
        outputs = {p.name:dict(sha256=digest(p),bytes=p.stat().st_size) for p in out.iterdir() if p.is_file()}
        write_json(out/'output_manifest.json', outputs)
        print(json.dumps(report,indent=2),flush=True)
        return report
    except Exception as exc:
        write_json(out/'FAILED.json', dict(error=str(exc),type=type(exc).__name__,config=asdict(config)))
        raise



## 4. Run the cleaning and validation loop
Each table repeats normalization, rejection, duplicate checks and sorting until its content hash stops changing. A pass cap raises an error if stability is not reached. Re-running this cell reads the immutable downloads again and creates a new output version.

No missing prices are interpolated, no returns are winsorized, no conflicting SEC values are averaged, and no corporate actions or historical identities are assumed verified. Returns are calculated only between consecutive exchange sessions. Split-close returns and provider-all-adjusted returns are separate columns. SEC long tables preserve taxonomies, units, period starts/ends, fiscal labels, accessions and amendments.


In [ ]:
report = run(Config(root=str(ROOT), price_version=PRICE_VERSION, sec_version=SEC_VERSION,
                    start=START, end=END, max_passes=MAX_PASSES,
                    return_review_threshold=LARGE_RETURN_THRESHOLD))
OUTPUT = Path(report['output_dir'])
print('Saved:', OUTPUT)


## 5. Review the results
`prices_clean.csv` includes SPY. `spy_clean.csv` is a convenience subset; do not append it to the prices table.

`sec_facts_clean_long.csv` contains all structurally valid facts in the snapshot. `sec_facts_study_asof.csv` restricts period ends to the study range and excludes information whose availability is after the study cutoff. It preserves repeated disclosures from different filings. Facts with period ends after their filing date can be prospective disclosures: they are retained and flagged in the complete long table, and excluded from the historical study view. **Do not join these facts to daily prices on period-end alone.** Choose a concept, unit, duration and filing known at the decision time first.

Exact acceptance timestamps are used when present in the downloaded submissions. Otherwise availability is approximated as the start of the next calendar day in New York, explicitly flagged. Referenced older submission chunks were not downloaded by the original collector. Current SEC extracts can contain revised historical values, so the output does not claim full point-in-time reconstruction.


In [ ]:
from IPython.display import display
summary_keys = ['status','core_ready','clean_price_rows','price_rows_quarantined',
                'missing_requested_sessions','large_return_reviews','sec_companies',
                'clean_sec_facts','study_asof_sec_facts','quarantined_sec_facts']
display(pd.DataFrame({'metric':summary_keys,'value':[report[k] for k in summary_keys]}))
display(pd.read_csv(OUTPUT / 'price_coverage.csv'))
display(pd.read_csv(OUTPUT / 'sec_coverage.csv'))
display(pd.read_csv(OUTPUT / 'cleaning_passes.csv').groupby('table').tail(1))
print('Unresolved evidence:')
for item in report['unresolved']:
    print('-', item)


## 6. Inspect exceptions before another pass
Corrections belong in the source collection or in a documented, separately versioned transformation. Leave original downloads and checksums intact. When new evidence or corrected downloads arrive, update the selected snapshot and rerun. Repeating the same automatic rules cannot establish missing issuer history or corporate-action evidence.

The default 25% absolute adjusted return threshold is a configurable review rule, not proof of an error. Missing leading history can reflect instrument changes and must not be filled merely to balance the panel. All output rows remain `core_ready=False` while source-level evidence is unresolved.


In [ ]:
for name in ['missing_sessions.csv','price_review.csv','prices_quarantine.csv',
             'sec_facts_quarantine.csv','issuer_identity_review.csv']:
    print('\n' + name)
    display(pd.read_csv(OUTPUT / name, nrows=20, dtype={'cik':'string','current_mapping_cik':'string'}))
# Large facts output: read selected columns or chunks rather than loading it all.
fact_sample = pd.read_csv(OUTPUT / 'sec_facts_study_asof.csv', nrows=20,
                          dtype={'cik':'string'})
display(fact_sample)


## Output guide and source definitions
- `cleaning_passes.csv`: per-table pass counts, rejected rows and stability.
- `prices_quarantine.csv` and `sec_facts_quarantine.csv`: rejected observations with reasons and source row references.
- `missing_sessions.csv` and `price_coverage.csv`: exchange-calendar coverage across the requested window.
- `price_review.csv`: large returns and zero-volume observations retained for review.
- `issuer_identity_review.csv`: SEC CIK checks, current ticker mapping and unresolved historical identity, including the collector's fixed XOM predecessor mapping.
- `sec_submissions.csv`: normalized downloaded recent filings. Extra response columns are preserved as JSON.
- `sec_older_submission_files_not_downloaded.csv`: referenced older chunks requiring further collection.
- `input_manifest.json`, `output_manifest.json`, `report.json`: reproducibility, checksums, selected versions, environment and unresolved items. If a run fails, `FAILED.json` marks its output folder as incomplete. Use only runs with `report.json`.

Definitions checked against the [Alpaca historical bars API](https://docs.alpaca.markets/us/reference/stockbars) and [SEC EDGAR API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces). Provider `adjustment=all` includes spin-off adjustments; the cleaner therefore labels its result as provider-all-adjusted, with corporate-action verification still outstanding.


## 7. Create both targets in one modeling CSV

The default risk definition is annualized volatility of the next 20 trading-session simple adjusted returns. Classification uses low/medium/high tertiles learned only from training rows. Prediction assumes the current session has closed. The same calendar dates define train/validation/test for both tasks. Twenty-session label windows crossing split boundaries are purged. Incomplete future windows remain unlabeled. SPY is a benchmark feature, and the modeled rows are the 50 companies. This creates a price-based table; SEC predictors have not been joined.

**Exclude both targets from X for every model.** Classification is derived from the regression target, so either target would leak the other. Also exclude future-window dates and split metadata. Use the explicit feature allowlist below. Fit feature preprocessing on training rows only, and do not use random cross-validation. If you change the training cutoff or use time-series cross-validation, recompute class thresholds on that fold's training labels and purge its boundaries.


In [ ]:
"""Create two future-risk targets on one shared, chronologically split table.
Run: python create_targets.py --prices data/cleaned/<run_id>/prices_clean.csv
"""
import argparse
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import exchange_calendars as xc

TARGETS = ['risk_volatility_next_20d', 'risk_class_next_20d']
# Deliberately narrow feature list. Targets, future dates and metadata are excluded.
FEATURES = ['open','high','low','close','volume','adjusted_close',
            'split_close_return','provider_all_adjusted_return','spy_return']


def future_volatility(returns, horizon=20):
    """At t, sample standard deviation of r[t+1:t+h+1], annualized."""
    return returns.rolling(horizon, min_periods=horizon).std(ddof=1).shift(-horizon) * np.sqrt(252)


def classify(values, low, high):
    result = pd.Series(pd.NA, index=values.index, dtype='string')
    valid = values.notna()
    result.loc[valid & values.le(low)] = 'low'
    result.loc[valid & values.gt(low) & values.le(high)] = 'medium'
    result.loc[valid & values.gt(high)] = 'high'
    return result


def build_targets(prices_path, output_dir=None, horizon=20):
    if horizon != 20:
        raise ValueError('This version names its targets next_20d and requires horizon=20.')
    prices_path = Path(prices_path).resolve()
    df = pd.read_csv(prices_path, dtype={'ticker':'string','session_date':'string'})
    if df.duplicated(['ticker','session_date']).any():
        raise ValueError('Duplicate ticker/date keys.')
    required = {'ticker','session_date','adjusted_close',*FEATURES[:-1]}
    if not required.issubset(df.columns):
        raise ValueError(f'Missing columns: {required-set(df.columns)}')
    if not df.ticker.eq('SPY').any():
        raise ValueError('SPY benchmark missing.')
    start, end = pd.to_datetime(df.session_date).agg(['min','max'])
    cal = xc.get_calendar('XNYS',start=start-pd.Timedelta(days=10),end=end+pd.Timedelta(days=10))
    sessions = cal.sessions_in_range(start,end).tz_localize(None).normalize()
    if len(sessions) < horizon * 5:
        raise ValueError('Insufficient sessions for three chronological splits.')
    if not pd.to_datetime(df.session_date).isin(sessions).all():
        raise ValueError('Input contains unexpected exchange dates.')
    train_boundary = int(len(sessions)*0.70)
    test_boundary = int(len(sessions)*0.85)
    validation_start, test_start = sessions[train_boundary], sessions[test_boundary]
    label_end_map = pd.Series(sessions, index=sessions).shift(-horizon)
    spy = df[df.ticker.eq('SPY')].copy()
    spy.index = pd.to_datetime(spy.session_date)
    spy_returns = spy.adjusted_close.reindex(sessions).pct_change(fill_method=None)
    parts = []
    # SPY provides a benchmark feature. The modeled instruments are the companies.
    for ticker, block in df[~df.ticker.eq('SPY')].groupby('ticker', sort=True):
        block = block.sort_values('session_date').copy()
        dates = pd.DatetimeIndex(pd.to_datetime(block.session_date))
        close = pd.Series(block.adjusted_close.to_numpy(),index=dates).reindex(sessions)
        if (close.dropna() <= 0).any() or not np.isfinite(close.dropna()).all():
            raise ValueError(f'{ticker}: invalid adjusted close')
        returns = close.pct_change(fill_method=None)
        vol = future_volatility(returns,horizon)
        block[TARGETS[0]] = vol.reindex(dates).to_numpy()
        block['target_window_end'] = label_end_map.reindex(dates).dt.strftime('%Y-%m-%d').to_numpy()
        block['spy_return'] = spy_returns.reindex(dates).to_numpy()
        parts.append(block)
    model = pd.concat(parts,ignore_index=True).sort_values(['ticker','session_date']).reset_index(drop=True)
    dates = pd.to_datetime(model.session_date)
    label_ends = pd.to_datetime(model.target_window_end)
    model['dataset_split'] = np.select([dates.lt(validation_start),dates.lt(test_start)], ['train','validation'],default='test')
    # Remove boundary rows from fitting/evaluation when their labels touch the next split.
    purge = (model.dataset_split.eq('train') & label_ends.ge(validation_start)) | (model.dataset_split.eq('validation') & label_ends.ge(test_start))
    model.loc[purge,'dataset_split'] = 'purged'
    missing = model[TARGETS[0]].isna()
    model.loc[missing,'dataset_split'] = 'unlabeled'
    training = model.loc[model.dataset_split.eq('train'),TARGETS[0]]
    if len(training) < 30:
        raise ValueError('Insufficient labeled training rows.')
    low, high = training.quantile([1/3,2/3]).to_numpy()
    if not np.isfinite([low,high]).all() or low >= high:
        raise ValueError('Training tertiles do not define distinct class boundaries.')
    model[TARGETS[1]] = classify(model[TARGETS[0]],float(low),float(high))
    if output_dir is None:
        root = prices_path.parents[3]
        output_dir = root/'data/modeling'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = Path(output_dir).resolve()
    out.mkdir(parents=True,exist_ok=False)
    destination = out/'model_dataset.csv'
    model.to_csv(destination,index=False)
    metadata = dict(output_csv=str(destination),source_csv=str(prices_path),
        source_sha256=hashlib.sha256(prices_path.read_bytes()).hexdigest(),
        target_columns=TARGETS,feature_columns=FEATURES,
        decision_time='After session t closes; features use data through t.',
        volatility_definition='sample_std(simple adjusted returns t+1 through t+20, ddof=1) * sqrt(252)',
        units='Annualized decimal volatility; 0.30 means 30 percent annualized volatility.',
        classification_definition='Pooled company training-row tertiles, fixed for validation/test.',
        class_boundaries=dict(low_upper_inclusive=float(low),medium_upper_inclusive=float(high)),
        validation_start=validation_start.strftime('%Y-%m-%d'),test_start=test_start.strftime('%Y-%m-%d'),
        split_definition='Shared XNYS sessions: first 70 percent train, next 15 percent validation, final 15 percent test; purge labels crossing boundaries.',
        rows=len(model),companies=int(model.ticker.nunique()),
        split_counts={str(k):int(v) for k,v in model.dataset_split.value_counts().items()},
        class_counts={str(k):int(v) for k,v in model[TARGETS[1]].dropna().value_counts().items()},
        unlabeled_rows=int(missing.sum()),purged_rows=int(purge.sum()),
        excluded_from_features=TARGETS+['target_window_end','dataset_split','source_row','dataset_version','ticker','session_date'],
        notes=['Both targets must be excluded from X for every model.',
               'Purged and unlabeled rows must not enter model fitting or evaluation.',
               'SPY is excluded as a modeled stock and supplies a contemporaneous benchmark feature.',
               'No SEC features joined; this is the price-based modeling table.',
               'Source corporate-action and identity verification remains unresolved; core_ready is unchanged.',
               'The target is future price volatility, not a Shariah-compliance or fundamental-risk score.'])
    # Independently verify the numeric label on first/middle/last eligible rows per issuer.
    for ticker, block in model.groupby('ticker'):
        source = df[df.ticker.eq(ticker)].copy()
        source.index = pd.to_datetime(source.session_date)
        daily = source.adjusted_close.reindex(sessions).pct_change(fill_method=None)
        eligible = block[block[TARGETS[0]].notna()]
        for ix in sorted(set([0,len(eligible)//2,len(eligible)-1])):
            if ix < 0:
                continue
            row = eligible.iloc[ix]
            pos = sessions.get_loc(pd.Timestamp(row.session_date))
            window = daily.iloc[pos+1:pos+horizon+1].to_numpy()
            assert len(window)==horizon and np.isfinite(window).all()
            assert np.isclose(row[TARGETS[0]],np.std(window,ddof=1)*np.sqrt(252),rtol=1e-9,atol=1e-12)
    assert model[TARGETS[0]].isna().equals(model[TARGETS[1]].isna())
    assert not set(TARGETS).intersection(FEATURES)
    assert model.loc[model.dataset_split.eq('train'),'target_window_end'].lt(validation_start.strftime('%Y-%m-%d')).all()
    assert model.loc[model.dataset_split.eq('validation'),'target_window_end'].lt(test_start.strftime('%Y-%m-%d')).all()
    saved = pd.read_csv(destination)
    assert len(saved)==len(model) and set(TARGETS).issubset(saved.columns)
    metadata['output_sha256'] = hashlib.sha256(destination.read_bytes()).hexdigest()
    metadata['verification'] = 'Passed independent forward-window calculations, split isolation, feature exclusion and CSV readback.'
    (out/'targets_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    (out/'README.md').write_text('''# One table, two targets

Use model_dataset.csv for both regression and classification. Both targets and all future-window metadata are excluded from feature_columns in targets_metadata.json. Select the desired target as y; never include the other target in X. Use dataset_split to select train, validation and test rows. Exclude purged and unlabeled rows. Null targets mean insufficient complete future returns and are not zero risk.

Risk is annualized next-20-session realized volatility, in decimal units. Classes use training-only pooled tertiles with fixed thresholds. SPY is a benchmark feature, not a modeled company. Targets use future data intentionally; features must not. Rows assume prediction after the current close. The source's unresolved corporate-action and historical identity checks remain applicable.
''',encoding='utf-8')
    print(json.dumps(metadata,indent=2))
    return destination


In [ ]:
MODEL_CSV = build_targets(OUTPUT / 'prices_clean.csv')
model_df = pd.read_csv(MODEL_CSV)
metadata = json.loads((MODEL_CSV.parent / 'targets_metadata.json').read_text())
feature_columns = metadata['feature_columns']
assert not set(metadata['target_columns']).intersection(feature_columns)

# The SAME X is used for either task. Neither target appears in X.
X = {}
y_regression = {}
y_classification = {}
for split in ['train', 'validation', 'test']:
    rows = model_df.loc[model_df['dataset_split'].eq(split)]
    X[split] = rows[feature_columns].copy()
    y_regression[split] = rows['risk_volatility_next_20d'].copy()
    y_classification[split] = rows['risk_class_next_20d'].copy()
    assert y_regression[split].notna().all()
    assert y_classification[split].notna().all()
    print(split, len(rows), 'rows')

# Regression model: fit(X['train'], y_regression['train'])
# Classification model: fit(X['train'], y_classification['train'])
# Use training-fitted preprocessing for validation and test; some first-day return
# features are missing. Any imputation must be fitted on training features only.
print('Modeling CSV:', MODEL_CSV)
print('Class boundaries:', metadata['class_boundaries'])
